In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
from psiop import *
from wkb import *

## 1D harmonic oscillator

In [ ]:
# =============================================================================
# 1D QUANTUM HARMONIC OSCILLATOR — WKB ANALYSIS
# =============================================================================
# Symbol:  p(x, ξ) = ξ² + x²   (harmonic oscillator, ω = 1)
# Ansatz:  u(x) ≈ exp(i S(x)/ε) · [a₀ + ε a₁ + ε² a₂]
#
# FIX 1 — Higher-order amplitudes:
#   For p = ξ² + x², the cross-derivative ∂²p/∂ξ∂x = 0, so the transport
#   equation feeds nothing into a₁ from a₀.  To make higher-order corrections
#   visible we seed a₁ with a non-trivial initial profile and use a symbol
#   with a weak perturbation:  p = ξ² + x² + δ·x·ξ  (δ = 0.15).
#   This breaks the pure harmonic structure and generates genuine a₁, a₂.
#
# FIX 2 — Spurious caustics:
#   caustic_correction='none' for the order-comparison and ε-sweep runs.
#   Caustics are shown separately with a dedicated tight threshold.
#
# FIX 3 — Normalisation:
#   ‖u‖ is dominated by rapid phase oscillations at small ε.
#   We track ‖a_total‖ (amplitude without phase) as the normalisation metric.
# =============================================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sympy import symbols

# ---------------------------------------------------------------------------
# SYMBOLS
# ---------------------------------------------------------------------------

x_sym, xi_sym = symbols('x xi', real=True)

# Perturbed symbol: adds cross-term δ·x·ξ so ∂²p/∂ξ∂x = δ ≠ 0
# → non-zero transport coupling → a₁, a₂ genuinely non-zero
DELTA   = 0.15
symbol  = xi_sym**2 + x_sym**2 + DELTA * x_sym * xi_sym

# Pure symbol for reference (no perturbation)
symbol_pure = xi_sym**2 + x_sym**2

# ---------------------------------------------------------------------------
# SETUP
# ---------------------------------------------------------------------------

DOMAIN      = (-4.0, 4.0)
RESOLUTION  = 500
N_RAYS      = 80

x0 = np.linspace(-3.0, 3.0, N_RAYS)

# Normalised Gaussian a₀
a0_raw  = np.exp(-x0**2 / 2.0)
norm_a0 = np.sqrt(np.trapezoid(a0_raw**2, x0))
a0_init = a0_raw / norm_a0

# Seed a₁ with a small antisymmetric perturbation so corrections are visible
a1_init = 0.05 * x0 * np.exp(-x0**2 / 2.0)

initial = {
    'x'  : x0,
    'p_x': np.ones(N_RAYS),
    'S'  : 0.5 * x0**2,
    'a'  : {
        0: a0_init,
        1: a1_init,
        2: np.zeros(N_RAYS),
    }
}

# ---------------------------------------------------------------------------
# EXACT REFERENCE  (ground state of pure harmonic oscillator at ε = 1)
# ---------------------------------------------------------------------------

x_ref     = np.linspace(DOMAIN[0], DOMAIN[1], RESOLUTION)
psi_exact = (1.0 / np.pi)**0.25 * np.exp(-x_ref**2 / 2.0)

# ---------------------------------------------------------------------------
# PART 1 — ORDER COMPARISON  (fixed ε = 0.05, no spurious caustic correction)
# ---------------------------------------------------------------------------

EPSILON_FIXED = 0.05
print("=" * 65)
print(f"  PART 1 · Order comparison   ε = {EPSILON_FIXED}")
print("=" * 65)

solutions_by_order = {}
for order in range(3):
    print(f"\n  ── Order {order} ──")
    solutions_by_order[order] = wkb_approximation(
        symbol, initial,
        order              = order,
        domain             = DOMAIN,
        resolution         = RESOLUTION,
        epsilon            = EPSILON_FIXED,
        caustic_correction = 'none',   # FIX 2: suppress spurious caustics
    )

# ---------------------------------------------------------------------------
# PART 2 — EPSILON SWEEP  (order = 2, no spurious caustic correction)
# ---------------------------------------------------------------------------

EPSILONS    = [0.20, 0.10, 0.05, 0.02]
ORDER_SWEEP = 2

print("\n" + "=" * 65)
print(f"  PART 2 · ε sweep   order = {ORDER_SWEEP}")
print("=" * 65)

solutions_by_eps = {}
for eps in EPSILONS:
    print(f"\n  ── ε = {eps} ──")
    solutions_by_eps[eps] = wkb_approximation(
        symbol, initial,
        order              = ORDER_SWEEP,
        domain             = DOMAIN,
        resolution         = RESOLUTION,
        epsilon            = eps,
        caustic_correction = 'none',
    )

# Main solution used for amplitude decomposition
sol_main = solutions_by_order[2]

# ---------------------------------------------------------------------------
# PART 3 — NORMALISATION CHECK  (FIX 3: use ‖a_total‖, not ‖u‖)
# ---------------------------------------------------------------------------

norms_atotal = []
for eps in EPSILONS:
    sol = solutions_by_eps[eps]
    xg  = sol['x']
    norms_atotal.append(np.sqrt(np.trapezoid(np.abs(sol['a_total'])**2, xg)))

# ---------------------------------------------------------------------------
# VISUALIZATION
# ---------------------------------------------------------------------------

COLORS_ORDER = {0: '#e07b39', 1: '#3a86ff', 2: '#2ec4b6'}
COLORS_EPS   = plt.cm.plasma(np.linspace(0.15, 0.85, len(EPSILONS)))

fig = plt.figure(figsize=(18, 22))
fig.patch.set_facecolor('#f8f9fa')

gs_top = gridspec.GridSpec(3, 3, top=0.97, bottom=0.56, hspace=0.45, wspace=0.35)
gs_mid = gridspec.GridSpec(1, 3, top=0.52, bottom=0.36, hspace=0.1,  wspace=0.35)
gs_bot = gridspec.GridSpec(1, 3, top=0.31, bottom=0.04, hspace=0.1,  wspace=0.35)

def _style(ax, title, xlabel='x', ylabel=''):
    ax.set_title(title, fontsize=11, fontweight='bold', pad=6)
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.grid(True, alpha=0.25, linestyle='--')
    ax.set_xlim(DOMAIN)
    ax.tick_params(labelsize=9)

# ── ROW 1  ·  |u|, Re(u), Im(u)  by order ───────────────────────────────────
row1 = [
    (r'$|u|$',              lambda u: np.abs(u)),
    (r'$\mathrm{Re}(u)$',  lambda u: np.real(u)),
    (r'$\mathrm{Im}(u)$',  lambda u: np.imag(u)),
]
for col, (title, fn) in enumerate(row1):
    ax = fig.add_subplot(gs_top[0, col])
    for order in range(3):
        sol = solutions_by_order[order]
        ax.plot(sol['x'], fn(sol['u']), color=COLORS_ORDER[order],
                lw=2.0, label=f'Order {order}')
    if col == 0:
        # Scale exact ψ₀ to match WKB normalization convention
        scale = np.max(np.abs(solutions_by_order[0]['u'])) / np.max(psi_exact)
        ax.plot(x_ref, scale * psi_exact, 'k--', lw=1.5, alpha=0.5,
                label='Exact ψ₀ (scaled)')
    _style(ax, f'Order comparison · {title}', ylabel=title)
    ax.legend(fontsize=8, loc='upper right')

# ── ROW 2  ·  Phase S(x)  by order ──────────────────────────────────────────
ax_phase = fig.add_subplot(gs_top[1, :])
for order in range(3):
    sol = solutions_by_order[order]
    ax_phase.plot(sol['x'], sol['S'], color=COLORS_ORDER[order],
                  lw=2.0, label=f'Order {order}')
ax_phase.plot(x_ref, 0.5 * x_ref**2, 'k--', lw=1.5, alpha=0.5,
              label='S = x²/2  (unperturbed eikonal)')
_style(ax_phase, f'Eikonal phase S(x)  [perturbed symbol δ = {DELTA}]',
       ylabel='S(x)')
ax_phase.legend(fontsize=9, loc='upper center', ncol=4)

# ── ROW 3  ·  ε sweep ────────────────────────────────────────────────────────
row3 = [
    (r'$|u|$',              lambda u: np.abs(u),   gs_top[2, 0]),
    (r'$\mathrm{Re}(u)$',  lambda u: np.real(u),  gs_top[2, 1]),
    (r'$\mathrm{Im}(u)$',  lambda u: np.imag(u),  gs_top[2, 2]),
]
for title, fn, gs_loc in row3:
    ax = fig.add_subplot(gs_loc)
    for i, eps in enumerate(EPSILONS):
        sol = solutions_by_eps[eps]
        ax.plot(sol['x'], fn(sol['u']), color=COLORS_EPS[i],
                lw=1.8, label=f'ε = {eps}')
    _style(ax, f'ε sweep (order {ORDER_SWEEP}) · {title}', ylabel=title)
    ax.legend(fontsize=8, loc='upper right')

# ── MIDDLE ROW  ·  Amplitude decomposition  aₖ ──────────────────────────────
titles_amp = [
    r'$a_0(x)$ — leading amplitude',
    r'$a_1(x)$ — first correction',
    r'$a_2(x)$ — second correction',
]
for col, k in enumerate(range(3)):
    ax = fig.add_subplot(gs_mid[0, col])
    ak     = sol_main['a'][k]
    weight = EPSILON_FIXED**k
    ax.plot(sol_main['x'], ak,          color='#3a86ff', lw=2.2,
            label=f'$a_{k}$')
    ax.plot(sol_main['x'], weight * ak, color='#e07b39', lw=2.0,
            linestyle='--',
            label=f'$\\varepsilon^{k} a_{k}$  (ε = {EPSILON_FIXED})')
    _style(ax, titles_amp[k], ylabel=f'$a_{k}$')
    ax.legend(fontsize=8)

# ── BOTTOM ROW  ·  contributions / norm / convergence ───────────────────────

# Panel A: stacked weighted contributions |εᵏ aₖ|
ax_a = fig.add_subplot(gs_bot[0, 0])
xg = sol_main['x']
for k in range(3):
    wk = EPSILON_FIXED**k
    ax_a.plot(xg, np.abs(wk * sol_main['a'][k]), color=COLORS_ORDER[k],
              lw=1.8, label=f'|ε^{k} a_{k}|')
ax_a.plot(xg, np.abs(sol_main['a_total']), 'k-', lw=2.2, label='|a_total|')
_style(ax_a, 'Weighted amplitude contributions', ylabel='magnitude')
ax_a.legend(fontsize=8)

# Panel B: ‖a_total‖ vs ε  (FIX 3: amplitude norm, not |u| norm)
ax_b = fig.add_subplot(gs_bot[0, 1])
ax_b.plot(EPSILONS, norms_atotal, 'o-', color='#3a86ff',
          lw=2, ms=8, label='‖a_total‖')
ax_b.axhline(1.0, color='k', lw=1.5, linestyle='--', alpha=0.6,
             label='Ideal = 1')
ax_b.set_xlabel('ε', fontsize=10)
ax_b.set_ylabel('‖a_total‖', fontsize=10)
ax_b.set_title('Amplitude norm vs ε\n(phase-independent normalisation)',
               fontsize=11, fontweight='bold', pad=6)
ax_b.grid(True, alpha=0.25, linestyle='--')
ax_b.legend(fontsize=9)
ax_b.invert_xaxis()

# Panel C: relative L² difference between consecutive orders
ax_c = fig.add_subplot(gs_bot[0, 2])
pairs    = [(0, 1), (1, 2)]
labels_c = ['‖u₁ – u₀‖/‖u₀‖', '‖u₂ – u₁‖/‖u₁‖']
colors_c = ['#e07b39', '#2ec4b6']
for (o1, o2), lbl, col in zip(pairs, labels_c, colors_c):
    s1 = solutions_by_order[o1]
    s2 = solutions_by_order[o2]
    rd = (np.linalg.norm(s2['u'] - s1['u']) /
          (np.linalg.norm(s1['u']) + 1e-12))
    ax_c.bar(lbl, rd, color=col, alpha=0.85, edgecolor='white', linewidth=1.2)
    ax_c.text(lbl, rd * 1.05, f'{rd:.2e}',
              ha='center', va='bottom', fontsize=9)
ax_c.set_ylabel('Relative L² difference', fontsize=10)
ax_c.set_title(f'Order convergence  (ε = {EPSILON_FIXED})',
               fontsize=11, fontweight='bold', pad=6)
ax_c.grid(True, alpha=0.25, linestyle='--', axis='y')
ax_c.tick_params(axis='x', labelsize=8)

# ── Global title ─────────────────────────────────────────────────────────────
fig.suptitle(
    '1D Quantum Harmonic Oscillator — WKB Analysis\n'
    r'$p(x,\xi) = \xi^2 + x^2 + \delta\, x\xi$'
    f',  δ = {DELTA}'
    r',   $u \approx e^{iS/\varepsilon}(a_0 + \varepsilon a_1 + \varepsilon^2 a_2)$',
    fontsize=13, fontweight='bold', y=0.995
)

plt.savefig('wkb_harmonic_oscillator.png', dpi=150,
            bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

# ---------------------------------------------------------------------------
# CONSOLE SUMMARY
# ---------------------------------------------------------------------------
print("\n" + "=" * 65)
print("  SUMMARY")
print("=" * 65)
print(f"\n  Symbol: ξ² + x² + {DELTA}·xξ   (δ = {DELTA})")
print(f"  Grid: {RESOLUTION} pts on {DOMAIN},  {N_RAYS} rays")
print(f"  ε (Part 1) = {EPSILON_FIXED},  caustic_correction = 'none'")

print("\n  Max |u| by order:")
for o in range(3):
    print(f"    Order {o}: {np.max(np.abs(solutions_by_order[o]['u'])):.6f}")

print(f"\n  ‖a_total‖ by ε  (order {ORDER_SWEEP}, phase-independent norm):")
for eps, nrm in zip(EPSILONS, norms_atotal):
    print(f"    ε = {eps:.2f}:  ‖a_total‖ = {nrm:.6f}")

print(f"\n  Amplitude values at x ≈ 0  (ε = {EPSILON_FIXED}, order 2):")
idx0 = np.argmin(np.abs(sol_main['x']))
for k in range(3):
    wk = EPSILON_FIXED**k
    print(f"    ε^{k} · a_{k}(0) = {wk * sol_main['a'][k][idx0]:.6e}")

print("\n  Correction magnitude check:")
for o1, o2 in [(0,1),(1,2)]:
    s1, s2 = solutions_by_order[o1], solutions_by_order[o2]
    rd = np.linalg.norm(s2['u']-s1['u']) / (np.linalg.norm(s1['u'])+1e-12)
    print(f"    ‖u_{o2} – u_{o1}‖ / ‖u_{o1}‖ = {rd:.4e}")

## 2D wave equation

In [ ]:
# =============================================================================
# 2D ANISOTROPIC WAVE EQUATION — WKB ANALYSIS
# =============================================================================
# Symbol:  p(x, y, ξ, η) = ξ² + η² + δ·x·ξ + δ·y·η
#
# The symbol must have BOTH:
#   (a) momentum cross-coupling for a₁:  ∂²p/∂ξ∂η  or  ∂²p/∂ξ∂x ≠ 0
#   (b) spatial-momentum coupling for a₂:  ∂²p/∂x∂ξ + ∂²p/∂y∂η ≠ 0
#
# Symbol choice: p = ξ² + η² + δ·x·ξ + δ·y·η
#   → ∂²p/∂x∂ξ = δ,  ∂²p/∂y∂η = δ   (drives a₁ from a₀)
#   → coupling = 2δ ≠ 0               (drives a₂ from a₁)
#   → physically: spatially-varying anisotropy / GRIN medium
#
# This guarantees the full hierarchy a₀ >> a₁ >> a₂ is non-trivially active.
# =============================================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sympy import symbols

# ---------------------------------------------------------------------------
# SYMBOLS
# ---------------------------------------------------------------------------

x_sym, y_sym, xi_sym, eta_sym = symbols('x y xi eta', real=True)

# p = ξ² + η² + δ·x·ξ + δ·y·η
# ∂²p/∂x∂ξ = δ,  ∂²p/∂y∂η = δ  →  coupling = 2δ → drives a₁ AND a₂
DELTA  = 0.15
symbol = (xi_sym**2 + eta_sym**2
          + DELTA * x_sym * xi_sym
          + DELTA * y_sym * eta_sym)

# ---------------------------------------------------------------------------
# INITIAL DATA — point source on circle of radius r₀
# ---------------------------------------------------------------------------

DOMAIN     = ((-3.5, 3.5), (-3.5, 3.5))
RESOLUTION = (80, 80)
N_RAYS     = 48

r0    = 0.3
theta = np.linspace(0, 2 * np.pi, N_RAYS, endpoint=False)

x_init  = r0 * np.cos(theta)
y_init  = r0 * np.sin(theta)
px_init = np.cos(theta)
py_init = np.sin(theta)
S_init  = np.zeros(N_RAYS)

# Seed a₁ with angular modulation; a₂ starts at zero but will be driven
a0_init = np.ones(N_RAYS)
a1_init = 0.1 * np.cos(2 * theta)

initial = {
    'x'  : x_init,
    'y'  : y_init,
    'p_x': px_init,
    'p_y': py_init,
    'S'  : S_init,
    'a'  : {
        0: a0_init,
        1: a1_init,
        2: np.zeros(N_RAYS),
    }
}

# ---------------------------------------------------------------------------
# EXACT REFERENCE — 2D geometric-optics envelope: amplitude ∝ 1/√r
# ---------------------------------------------------------------------------

def exact_envelope(X, Y, r0=r0):
    r   = np.sqrt(X**2 + Y**2)
    env = np.where(r > r0 + 0.05,
                   1.0 / np.sqrt(np.maximum(r - r0, 1e-6)), 0.0)
    return env / (np.max(env) + 1e-12)

# ---------------------------------------------------------------------------
# PART 1 — ORDER COMPARISON  (fixed ε = 0.1)
# ---------------------------------------------------------------------------

EPSILON_FIXED = 0.1
ORDER_MAX     = 2

print("=" * 65)
print(f"  PART 1 · Order comparison   ε = {EPSILON_FIXED}")
print("=" * 65)

solutions_by_order = {}
for order in range(ORDER_MAX + 1):
    print(f"\n  ── Order {order} ──")
    solutions_by_order[order] = wkb_approximation(
        symbol, initial,
        order              = order,
        domain             = DOMAIN,
        resolution         = RESOLUTION,
        epsilon            = EPSILON_FIXED,
        caustic_correction = 'none',
    )

# ---------------------------------------------------------------------------
# PART 2 — EPSILON SWEEP  (order = 2)
# ---------------------------------------------------------------------------

EPSILONS    = [0.20, 0.10, 0.05, 0.02]
ORDER_SWEEP = 2

print("\n" + "=" * 65)
print(f"  PART 2 · ε sweep   order = {ORDER_SWEEP}")
print("=" * 65)

solutions_by_eps = {}
for eps in EPSILONS:
    print(f"\n  ── ε = {eps} ──")
    solutions_by_eps[eps] = wkb_approximation(
        symbol, initial,
        order              = ORDER_SWEEP,
        domain             = DOMAIN,
        resolution         = RESOLUTION,
        epsilon            = eps,
        caustic_correction = 'none',
    )

sol_main = solutions_by_order[ORDER_MAX]
X_main   = sol_main['x']
Y_main   = sol_main['y']

# ---------------------------------------------------------------------------
# PART 3 — NORMALISATION  (‖a_total‖₂ via 2D integration)
# ---------------------------------------------------------------------------

def norm2d(field, X, Y):
    dx = X[1, 0] - X[0, 0]
    dy = Y[0, 1] - Y[0, 0]
    return np.sqrt(np.sum(np.abs(field)**2) * dx * dy)

norms_atotal = []
for eps in EPSILONS:
    sol = solutions_by_eps[eps]
    norms_atotal.append(norm2d(sol['a_total'], sol['x'], sol['y']))

# ---------------------------------------------------------------------------
# VISUALIZATION
# ---------------------------------------------------------------------------

COLORS_ORDER = {0: '#e07b39', 1: '#3a86ff', 2: '#2ec4b6'}
COLORS_EPS   = plt.cm.plasma(np.linspace(0.15, 0.85, len(EPSILONS)))

fig = plt.figure(figsize=(20, 26))
fig.patch.set_facecolor('#f8f9fa')

gs_ord = gridspec.GridSpec(ORDER_MAX + 1, 3, top=0.97, bottom=0.70,
                            hspace=0.40, wspace=0.30)
gs_eps = gridspec.GridSpec(2, 4,          top=0.66, bottom=0.50,
                            hspace=0.35, wspace=0.30)
gs_amp = gridspec.GridSpec(1, 3,          top=0.46, bottom=0.31,
                            hspace=0.1,  wspace=0.30)
gs_bot = gridspec.GridSpec(1, 3,          top=0.27, bottom=0.02,
                            hspace=0.1,  wspace=0.30)

def _style2d(ax, title, rays=None, n_ray_lines=12):
    ax.set_title(title, fontsize=10, fontweight='bold', pad=5)
    ax.set_xlabel('x', fontsize=9)
    ax.set_ylabel('y', fontsize=9)
    ax.set_aspect('equal')
    ax.tick_params(labelsize=8)
    if rays is not None:
        step = max(1, len(rays) // n_ray_lines)
        for ray in rays[::step]:
            ax.plot(ray['x'], ray['y'], 'w-', alpha=0.2, lw=0.7)

# ── ROWS 1–3  ·  one row per order, columns = |u|, Re(u), Im(u) ─────────────
panel_fns = [
    (r'$|u|$',             lambda u: np.abs(u),   'viridis'),
    (r'$\mathrm{Re}(u)$', lambda u: np.real(u),  'RdBu_r'),
    (r'$\mathrm{Im}(u)$', lambda u: np.imag(u),  'RdBu_r'),
]
for row, order in enumerate(range(ORDER_MAX + 1)):
    sol  = solutions_by_order[order]
    X, Y = sol['x'], sol['y']
    for col, (comp_title, fn, cmap) in enumerate(panel_fns):
        ax   = fig.add_subplot(gs_ord[row, col])
        data = fn(sol['u'])
        vmax = np.nanpercentile(np.abs(data), 98)
        vmin = -vmax if cmap == 'RdBu_r' else 0
        im   = ax.pcolormesh(X, Y, data, shading='auto',
                             cmap=cmap, vmin=vmin, vmax=vmax)
        plt.colorbar(im, ax=ax, shrink=0.85, pad=0.02)
        _style2d(ax, f'Order {order} · {comp_title}', rays=sol['rays'])

# ── ROWS 4–5  ·  ε sweep: |u| (top row) and Re(u) (bottom row) ──────────────
for col, eps in enumerate(EPSILONS):
    sol  = solutions_by_eps[eps]
    X, Y = sol['x'], sol['y']

    ax_top = fig.add_subplot(gs_eps[0, col])
    vmax   = np.nanpercentile(np.abs(sol['u']), 98)
    im1    = ax_top.pcolormesh(X, Y, np.abs(sol['u']),
                                shading='auto', cmap='viridis',
                                vmin=0, vmax=vmax)
    plt.colorbar(im1, ax=ax_top, shrink=0.85, pad=0.02)
    _style2d(ax_top, f'ε = {eps} · $|u|$', rays=sol['rays'])

    ax_bot = fig.add_subplot(gs_eps[1, col])
    vmax_r = np.nanpercentile(np.abs(np.real(sol['u'])), 98)
    im2    = ax_bot.pcolormesh(X, Y, np.real(sol['u']),
                                shading='auto', cmap='RdBu_r',
                                vmin=-vmax_r, vmax=vmax_r)
    plt.colorbar(im2, ax=ax_bot, shrink=0.85, pad=0.02)
    _style2d(ax_bot, f'ε = {eps} · $\\mathrm{{Re}}(u)$', rays=sol['rays'])

# ── MIDDLE ROW  ·  Amplitude decomposition  aₖ(x,y) ────────────────────────
titles_amp = [
    r'$a_0$ — leading amplitude',
    r'$a_1$ — first correction',
    r'$a_2$ — second correction',
]
for col, k in enumerate(range(ORDER_MAX + 1)):
    ax   = fig.add_subplot(gs_amp[0, col])
    ak   = sol_main['a'][k]
    vmax = np.nanpercentile(np.abs(ak), 98) or 1e-10
    im   = ax.pcolormesh(X_main, Y_main, ak,
                         shading='auto', cmap='viridis', vmin=0, vmax=vmax)
    plt.colorbar(im, ax=ax, shrink=0.85, pad=0.02, label=f'$a_{k}$')
    if k == 0:
        ref = exact_envelope(X_main, Y_main)
        ax.contour(X_main, Y_main, ref, levels=5,
                   colors='white', alpha=0.5, linewidths=0.8, linestyles='--')
    weight_str = f'ε^{k}·max = {EPSILON_FIXED**k * np.max(np.abs(ak)):.3e}'
    _style2d(ax, f'{titles_amp[k]}\n({weight_str})')

# ── BOTTOM ROW  ·  Phase S / norm vs ε / convergence ────────────────────────

# Panel A: eikonal phase S(x,y)
ax_s = fig.add_subplot(gs_bot[0, 0])
im_s = ax_s.pcolormesh(X_main, Y_main, sol_main['S'],
                        shading='auto', cmap='twilight')
plt.colorbar(im_s, ax=ax_s, shrink=0.85, pad=0.02, label='S(x,y)')
_style2d(ax_s, f'Eikonal phase S(x,y)  [δ = {DELTA}]', rays=sol_main['rays'])

# Panel B: ‖a_total‖₂ vs ε
ax_n = fig.add_subplot(gs_bot[0, 1])
ax_n.plot(EPSILONS, norms_atotal, 'o-', color='#3a86ff', lw=2, ms=8,
          label='‖a_total‖₂')
ax_n.axhline(norms_atotal[-1], color='k', lw=1.2,
             linestyle='--', alpha=0.5, label='ε → 0 limit')
ax_n.set_xlabel('ε', fontsize=10)
ax_n.set_ylabel('‖a_total‖₂', fontsize=10)
ax_n.set_title('Amplitude norm vs ε\n(phase-independent normalisation)',
               fontsize=10, fontweight='bold', pad=5)
ax_n.grid(True, alpha=0.25, linestyle='--')
ax_n.legend(fontsize=9)
ax_n.invert_xaxis()

# Panel C: relative L² convergence between consecutive orders
ax_c = fig.add_subplot(gs_bot[0, 2])
pairs    = [(0, 1), (1, 2)]
labels_c = ['‖u₁–u₀‖/‖u₀‖', '‖u₂–u₁‖/‖u₁‖']
colors_c = ['#e07b39', '#2ec4b6']
for (o1, o2), lbl, col in zip(pairs, labels_c, colors_c):
    s1, s2 = solutions_by_order[o1], solutions_by_order[o2]
    rd = (np.linalg.norm(s2['u'] - s1['u']) /
          (np.linalg.norm(s1['u']) + 1e-12))
    ax_c.bar(lbl, rd, color=col, alpha=0.85,
             edgecolor='white', linewidth=1.2)
    ax_c.text(lbl, rd * 1.05, f'{rd:.2e}',
              ha='center', va='bottom', fontsize=9)
ax_c.set_ylabel('Relative L² difference', fontsize=10)
ax_c.set_title(f'Order convergence  (ε = {EPSILON_FIXED})',
               fontsize=10, fontweight='bold', pad=5)
ax_c.grid(True, alpha=0.25, linestyle='--', axis='y')
ax_c.tick_params(axis='x', labelsize=8)

# ── Global title ─────────────────────────────────────────────────────────────
fig.suptitle(
    '2D Anisotropic Wave Equation — WKB Analysis\n'
    r'$p = \xi^2 + \eta^2 + \delta\,x\xi + \delta\,y\eta$'
    f',  δ = {DELTA}'
    r',   $u \approx e^{iS/\varepsilon}(a_0 + \varepsilon a_1 + \varepsilon^2 a_2)$',
    fontsize=13, fontweight='bold', y=0.995
)

plt.savefig('wkb_wave2d.png', dpi=150,
            bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

# ---------------------------------------------------------------------------
# CONSOLE SUMMARY
# ---------------------------------------------------------------------------
print("\n" + "=" * 65)
print("  SUMMARY")
print("=" * 65)
print(f"\n  Symbol: ξ² + η² + δ·xξ + δ·yη   (δ = {DELTA})")
print(f"  ∂²p/∂x∂ξ = ∂²p/∂y∂η = δ = {DELTA}  →  coupling drives a₁ AND a₂")
print(f"  Grid: {RESOLUTION} on {DOMAIN}")
print(f"  N_RAYS = {N_RAYS},  r₀ = {r0}")
print(f"  ε (Part 1) = {EPSILON_FIXED},  caustic_correction = 'none'")

print("\n  Max |u| by order:")
for o in range(ORDER_MAX + 1):
    print(f"    Order {o}: {np.max(np.abs(solutions_by_order[o]['u'])):.6f}")

print(f"\n  ‖a_total‖₂ by ε  (order {ORDER_SWEEP}):")
for eps, nrm in zip(EPSILONS, norms_atotal):
    print(f"    ε = {eps:.2f}:  ‖a_total‖₂ = {nrm:.6f}")

print(f"\n  Amplitude maxima  (ε = {EPSILON_FIXED}, order {ORDER_MAX}):")
for k in range(ORDER_MAX + 1):
    wk = EPSILON_FIXED**k
    print(f"    max|ε^{k} · a_{k}| = {wk * np.max(np.abs(sol_main['a'][k])):.6e}")

print("\n  Order convergence:")
for o1, o2 in [(0, 1), (1, 2)]:
    s1, s2 = solutions_by_order[o1], solutions_by_order[o2]
    rd = np.linalg.norm(s2['u'] - s1['u']) / (np.linalg.norm(s1['u']) + 1e-12)
    print(f"    ‖u_{o2} – u_{o1}‖ / ‖u_{o1}‖ = {rd:.4e}")

# psiOp and WKB

## WKB 1D – symbol ξ²

In [ ]:
# =============================================================================
# 1D WKB — Non-standard symbol: p(x, ξ) = ξ² + exp(-(x-ξ)²)
# =============================================================================
# This symbol couples position and momentum through the Gaussian exp(-(x-ξ)²),
# producing non-trivial ray bending and higher-order amplitude corrections.
#
# BUG FIX: amplitude arrays must share the same length as 'x'.
#   The original code built a0_init/a1_init from theta (n_pts=30 points)
#   while 'x' had only 10 points → shape mismatch (30,) vs (10,).
#   Solution: define N_RAYS once and use it everywhere consistently.
# =============================================================================

import numpy as np
import matplotlib.pyplot as plt
from sympy import symbols, exp

x_sym, xi_sym = symbols('x xi')
symbol = xi_sym**2 + exp(-(x_sym - xi_sym)**2)

# ---------------------------------------------------------------------------
# PARAMETERS
# ---------------------------------------------------------------------------

D           = (-6, 6)
ORDER       = 3
EPSILON     = 0.5
N_RAYS      = 30        # single source of truth for all array lengths

# ---------------------------------------------------------------------------
# INITIAL CONDITIONS  —  all arrays of length N_RAYS
# ---------------------------------------------------------------------------
# Angular variable for modulating the amplitudes (purely a parametrisation
# of the N_RAYS initial points, unrelated to 2D geometry)
theta = np.linspace(0, 2 * np.pi, N_RAYS, endpoint=False)

x_init  = np.linspace(D[0], D[1], N_RAYS)   # FIX: was np.linspace(..., 10)
S_init  = np.zeros(N_RAYS)
px_init = np.ones(N_RAYS)

a0_init = 1.0 + 0.3 * np.cos(3 * theta)     # shape (N_RAYS,) ✓
a1_init = 0.1 * np.sin(3 * theta)            # shape (N_RAYS,) ✓

initial_phase = {
    'x'  : x_init,
    'S'  : S_init,
    'p_x': px_init,
    'a'  : {
        0: a0_init,
        1: a1_init,
        2: np.zeros(N_RAYS),
        3: np.ones(N_RAYS),
    }
}

# ---------------------------------------------------------------------------
# PART 1 — Full solution with caustic correction
# ---------------------------------------------------------------------------

print("=" * 65)
print("  PART 1 · Full solution with caustic correction")
print("=" * 65)

res = wkb_approximation(
    symbol, initial_phase,
    order              = ORDER,
    domain             = D,
    resolution         = 400,
    epsilon            = 0.05,             # tight ε for the corrected run
    caustic_correction = 'auto',
    caustic_threshold  = EPSILON,          # = 0.5, intentionally loose
)

# ---------------------------------------------------------------------------
# PART 2 — Order comparison  (ε = EPSILON = 0.5)
# ---------------------------------------------------------------------------

print("\n" + "=" * 65)
print(f"  PART 2 · Order comparison   ε = {EPSILON}")
print("=" * 65)

sols, fig_compare = compare_orders(
    symbol, initial_phase,
    max_order  = ORDER,
    domain     = D,
    resolution = 300,
    epsilon    = EPSILON,
)

# ---------------------------------------------------------------------------
# PART 3 — Amplitude decomposition and phase space  (highest order)
# ---------------------------------------------------------------------------

print("\n" + "=" * 65)
print(f"  PART 3 · Amplitude decomposition and phase space  (order {ORDER})")
print("=" * 65)

fig_amp   = plot_amplitude_decomposition(sols[ORDER])
fig_phase = plot_phase_space(sols[ORDER])

# ---------------------------------------------------------------------------
# PART 4 — Caustic analysis plots
# ---------------------------------------------------------------------------

print("\n" + "=" * 65)
print("  PART 4 · Caustic plots")
print("=" * 65)

fig_caustic      = plot_with_caustics(res, component='abs', highlight_caustics=True)
fig_caustic_anal = plot_caustic_analysis(res)

plt.show()

# ---------------------------------------------------------------------------
# CONSOLE SUMMARY
# ---------------------------------------------------------------------------

print("\n" + "=" * 65)
print("  SUMMARY")
print("=" * 65)
print(f"  Symbol:   ξ² + exp(-(x-ξ)²)")
print(f"  N_RAYS:   {N_RAYS}  (consistent across x, S, p_x, a0..a3)")
print(f"  Domain:   {D}")
print(f"  Orders compared: 0 → {ORDER}")
print(f"  ε (compare_orders): {EPSILON}")
print(f"  ε (caustic run):    0.05")

print("\n  Max |u| by order:")
for o in range(ORDER + 1):
    print(f"    Order {o}: {np.max(np.abs(sols[o]['u'])):.6f}")

print(f"\n  Caustics detected: {len(res.get('caustics', []))}")
print(f"  Caustic correction: {res.get('caustic_correction', 'none')}")
res = wkb_approximation(
    symbol, initial_phase,
    order              = ORDER,
    domain             = D,
    resolution         = 400,
    epsilon            = EPSILON,
    caustic_correction = 'auto',
    caustic_threshold  = 1e-2,     # tight threshold — physical caustics only
)

# ---------------------------------------------------------------------------
# PART 2 — Order comparison  (same ε = EPSILON throughout)
# ---------------------------------------------------------------------------

print("\n" + "=" * 65)
print(f"  PART 2 · Order comparison   ε = {EPSILON}")
print("=" * 65)

sols, fig_compare = compare_orders(
    symbol, initial_phase,
    max_order  = ORDER,
    domain     = D,
    resolution = 300,
    epsilon    = EPSILON,          # BUG B fix: same ε as the main run
)

# ---------------------------------------------------------------------------
# PART 3 — Amplitude decomposition and phase space  (order = ORDER)
# ---------------------------------------------------------------------------

print("\n" + "=" * 65)
print(f"  PART 3 · Amplitude decomposition and phase space  (order {ORDER})")
print("=" * 65)

fig_amp   = plot_amplitude_decomposition(sols[ORDER])
fig_phase = plot_phase_space(sols[ORDER])

# ---------------------------------------------------------------------------
# PART 4 — Caustic plots
# ---------------------------------------------------------------------------

print("\n" + "=" * 65)
print("  PART 4 · Caustic plots")
print("=" * 65)

fig_caustic1 = plot_with_caustics(res, component='abs', highlight_caustics=True)
fig_caustic2 = plot_caustic_analysis(res)

# ---------------------------------------------------------------------------
# CONVERGENCE SUMMARY
# ---------------------------------------------------------------------------

print("\n" + "=" * 65)
print("  SUMMARY")
print("=" * 65)
print(f"  Symbol:   ξ² + exp(-(x-ξ)²)")
print(f"  N_RAYS:   {N_RAYS}  (consistent across x, S, p_x, a0..a3)")
print(f"  Domain:   {D}")
print(f"  Orders compared: 0 → {ORDER}")
print(f"  ε (all runs): {EPSILON}")

print("\n  Max |u| by order:")
for o in range(ORDER + 1):
    print(f"    Order {o}: {np.max(np.abs(sols[o]['u'])):.6f}")

print(f"\n  Caustics detected: {len(res.get('caustics', []))}")
print(f"  Caustic correction: {res.get('caustic_correction', 'none')}")

print("\n  Amplitude magnitudes at x ≈ 0  (order 3):")
sol3 = sols[ORDER]
idx0 = np.argmin(np.abs(sol3['x']))
for k in range(ORDER + 1):
    wk = EPSILON**k
    val = wk * sol3['a'][k][idx0]
    print(f"    ε^{k} · a_{k}(0) = {val:.6e}")

print("\n  Relative L² differences between consecutive orders:")
for o1, o2 in zip(range(ORDER), range(1, ORDER + 1)):
    u1 = sols[o1]['u']
    u2 = sols[o2]['u']
    rd = np.linalg.norm(u2 - u1) / (np.linalg.norm(u1) + 1e-12)
    flag = "  ← diverging!" if rd > 0.5 else ""
    print(f"    ‖u_{o2} – u_{o1}‖ / ‖u_{o1}‖ = {rd:.4e}{flag}")

plt.show()

In [ ]:
# =============================================================================
# WKB 1D  —  symbol  p(x, ξ) = ξ² + exp(-(x-ξ)²)
# =============================================================================
# This symbol has non-trivial x-ξ coupling through exp(-(x-ξ)²):
#   ∂²p/∂ξ∂x = 2(x-ξ)(1 - 2(x-ξ)²) · exp(-(x-ξ)²)   ≠ 0 generically
#   ∂³p/∂ξ³  is also non-zero → all correction orders are genuinely active.
#
# BUG A (fixed) — Size mismatch in original cell:
#   x/S/p_x had length 10, but a0..a3 were built from theta (length n_pts=30).
#   The fix: a single N_RAYS constant drives ALL arrays consistently.
#
# BUG B (fixed) — ε = 0.5 passed to compare_orders caused divergence.
#   The fix: one shared EPSILON used everywhere; convergence verified below.
# =============================================================================

import numpy as np
import matplotlib.pyplot as plt
from sympy import symbols, exp

# ---------------------------------------------------------------------------
# SYMBOL
# ---------------------------------------------------------------------------

x_sym, xi_sym = symbols('x xi', real=True)
symbol = xi_sym**2 + exp(-(x_sym - xi_sym)**2)

# ---------------------------------------------------------------------------
# CONSISTENT SETUP  (BUG A fix: single N_RAYS for every array)
# ---------------------------------------------------------------------------

DOMAIN   = (-6, 6)
N_RAYS   = 40        # same length for x, S, p_x AND all amplitude arrays
ORDER    = 3
EPSILON  = 0.05      # BUG B fix: small enough for the asymptotic series to converge

x0 = np.linspace(DOMAIN[0], DOMAIN[1], N_RAYS)

# Angular modulation expressed on the same N_RAYS grid
# (theta is just a phase parameter for building varied initial amplitudes)
theta = np.linspace(0, 2 * np.pi, N_RAYS, endpoint=False)

a0_init = 1.0 + 0.3 * np.cos(3 * theta)   # length N_RAYS ✓
a1_init = 0.1 * np.sin(3 * theta)          # length N_RAYS ✓
a2_init = np.zeros(N_RAYS)                 # length N_RAYS ✓
a3_init = 0.01 * np.cos(theta)             # length N_RAYS ✓  (small seed, not ones)

initial_phase = {
    'x'  : x0,                    # length N_RAYS ✓
    'S'  : np.zeros(N_RAYS),      # length N_RAYS ✓
    'p_x': np.ones(N_RAYS),       # length N_RAYS ✓
    'a'  : {
        0: a0_init,
        1: a1_init,
        2: a2_init,
        3: a3_init,
    }
}

# ---------------------------------------------------------------------------
# PART 1 — Full solution with caustic correction  (ε = EPSILON)
# ---------------------------------------------------------------------------

print("=" * 65)
print("  PART 1 · Full solution with caustic correction")
print("=" * 65)
CAUSTIC_THRESHOLD = 1e-2
res = wkb_approximation(
    symbol, initial_phase,
    order              = ORDER,
    domain             = DOMAIN,
    resolution         = 400,
    epsilon            = EPSILON,
    caustic_correction = 'auto',
    caustic_threshold  = CAUSTIC_THRESHOLD,     # tight threshold — physical caustics only
)

# ---------------------------------------------------------------------------
# PART 2 — Order comparison  (same ε = EPSILON throughout)
# ---------------------------------------------------------------------------

print("\n" + "=" * 65)
print(f"  PART 2 · Order comparison   ε = {EPSILON}")
print("=" * 65)

sols, fig_compare = compare_orders(
    symbol, initial_phase,
    max_order  = ORDER,
    domain     = DOMAIN,
    caustic_threshold  = CAUSTIC_THRESHOLD, 
    resolution = 300,
    epsilon    = EPSILON,          # BUG B fix: same ε as the main run
)

# ---------------------------------------------------------------------------
# PART 3 — Amplitude decomposition and phase space  (order = ORDER)
# ---------------------------------------------------------------------------

print("\n" + "=" * 65)
print(f"  PART 3 · Amplitude decomposition and phase space  (order {ORDER})")
print("=" * 65)

fig_amp   = plot_amplitude_decomposition(sols[ORDER])
fig_phase = plot_phase_space(sols[ORDER])

# ---------------------------------------------------------------------------
# PART 4 — Caustic plots
# ---------------------------------------------------------------------------

print("\n" + "=" * 65)
print("  PART 4 · Caustic plots")
print("=" * 65)

fig_caustic1 = plot_with_caustics(res, component='abs', highlight_caustics=True)
fig_caustic2 = plot_caustic_analysis(res)

# ---------------------------------------------------------------------------
# CONVERGENCE SUMMARY
# ---------------------------------------------------------------------------

print("\n" + "=" * 65)
print("  SUMMARY")
print("=" * 65)
print(f"  Symbol:   ξ² + exp(-(x-ξ)²)")
print(f"  N_RAYS:   {N_RAYS}  (consistent across x, S, p_x, a0..a3)")
print(f"  Domain:   {DOMAIN}")
print(f"  Orders compared: 0 → {ORDER}")
print(f"  ε (all runs): {EPSILON}")

print("\n  Max |u| by order:")
for o in range(ORDER + 1):
    print(f"    Order {o}: {np.max(np.abs(sols[o]['u'])):.6f}")

print(f"\n  Caustics detected: {len(res.get('caustics', []))}")
print(f"  Caustic correction: {res.get('caustic_correction', 'none')}")

print("\n  Amplitude magnitudes at x ≈ 0  (order 3):")
sol3 = sols[ORDER]
idx0 = np.argmin(np.abs(sol3['x']))
for k in range(ORDER + 1):
    wk = EPSILON**k
    val = wk * sol3['a'][k][idx0]
    print(f"    ε^{k} · a_{k}(0) = {val:.6e}")

print("\n  Relative L² differences between consecutive orders:")
for o1, o2 in zip(range(ORDER), range(1, ORDER + 1)):
    u1 = sols[o1]['u']
    u2 = sols[o2]['u']
    rd = np.linalg.norm(u2 - u1) / (np.linalg.norm(u1) + 1e-12)
    flag = "  ← diverging!" if rd > 0.5 else ""
    print(f"    ‖u_{o2} – u_{o1}‖ / ‖u_{o1}‖ = {rd:.4e}{flag}")

plt.show()

## Symbol: ξ² + η² - 1 (constant speed)

In [ ]:
x, y, xi, eta = symbols('x y xi eta', real=True)

# Symbol: ξ² + η² - 1 (constant speed)
p_2d = xi**2 + eta**2 - sin(x + y)

# Initial condition: line segment
# Circular initial curve
n_pts = 40
theta = np.linspace(0, 2*np.pi, n_pts, endpoint=False)
radius = 1

x_init = radius * np.cos(theta)
y_init = radius * np.sin(theta)

# Outward propagating wave
px_init = np.cos(theta)
py_init = np.sin(theta)

# Amplitude with angular modulation
a0_init = 1.0 + 0.3 * np.cos(3*theta)
a1_init = 0.1 * np.sin(3*theta)

ic_1 = {
    'x': x_init,
    'y': y_init,
    'S': np.zeros(n_pts),
    'p_x': px_init,
    'p_y': py_init,
    'a': {
        0: a0_init,
        1: a1_init,
        2: np.zeros(n_pts)
    }
}

ic_2 = create_initial_data_line(
    x_range=(-2, 2),
    n_points=20,
    direction=(-1, 1),
    y_intercept=-2.0
)

ic_3 = create_initial_data_circle(radius=1.0, n_points=30, outward=True)

ic_4 = create_initial_data_point_source(0, 0, n_rays=24)

ic = ic_4
order = 2
D = ((-20, 20), (-20, 20))
epsilon=0.05

sol = wkb_approximation(
    p_2d, ic,
    order=order,
    domain=D,
    resolution=100,
    epsilon=epsilon
)
# Compare different orders
sols = compare_orders(
    p_2d, ic,
    max_order=order,
    domain=D,
    resolution=120,
    epsilon=epsilon
)

# Plot amplitude decomposition for order 3
fig_amp = plot_amplitude_decomposition(sols[0][order])
# Phase space analysis
fig_phase = plot_phase_space(sols[0][order])
fig1 = plot_with_caustics(sol, component='abs', highlight_caustics=True)
fig2 = plot_caustic_analysis(sol)
plt.show()


## Isotropic wave equation

In [ ]:
x, y, xi, eta = symbols('x y xi eta', real=True)

# Isotropic wave equation
p = xi**2 + eta**2 - 1

# Initial curve: two converging arcs that create a cusp
n_pts = 30

# Upper arc
theta_upper = np.linspace(0.2*np.pi, 0.8*np.pi, n_pts//2)
r = 2.0
x_upper = r * np.cos(theta_upper)
y_upper = r * np.sin(theta_upper) - 1

# Lower arc
theta_lower = np.linspace(-0.8*np.pi, -0.2*np.pi, n_pts//2)
x_lower = r * np.cos(theta_lower)
y_lower = r * np.sin(theta_lower) + 1

x_init = np.concatenate([x_upper, x_lower])
y_init = np.concatenate([y_upper, y_lower])

# Rays pointing inward
px_init = -np.concatenate([np.cos(theta_upper), np.cos(theta_lower)])
py_init = -np.concatenate([np.sin(theta_upper), np.sin(theta_lower)])

# Normalize
p_mag = np.sqrt(px_init**2 + py_init**2)
px_init /= p_mag
py_init /= p_mag

ic = {
    'x': x_init,
    'y': y_init,
    'S': np.zeros(len(x_init)),
    'p_x': px_init,
    'p_y': py_init,
    'a': np.ones(len(x_init))
}

# Compute with caustic correction
sol_corrected = wkb_approximation(
    p, ic,
    order=1,
    domain=((-4, 4), (-4, 4)),
    resolution=100,
    epsilon=0.12,
    caustic_correction='auto',
    caustic_threshold=0.1
)

# Visualize
fig1 = plot_with_caustics(sol_corrected, component='abs', highlight_caustics=True)
fig2 = plot_caustic_analysis(sol_corrected)

## Maslov Index Tracking Through Multiple Caustics

In [ ]:
x, xi = symbols('x xi', real=True)

# Variable speed creating multiple caustics
# c(x) = 1 + 0.3*sin(x) causes focusing/defocusing
c = 1 + 0.3*sin(0.5*x)
p = xi**2 - c**2

# Single ray
ic = {
    'x': [0.0],
    'S': [0.0],
    'p_x': [1.0],
    'a': [1.0]
}

# Compute solution
sol = wkb_approximation(
    p, ic,
    order=1,
    domain=(-2, 20),
    resolution=500,
    epsilon=0.1,
    caustic_correction='maslov',
    caustic_threshold=0.01
)

# Plot with Maslov phases
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

x_grid = sol['x']

# 1. Amplitude
axes[0].plot(x_grid, np.abs(sol['u']), 'b-', linewidth=2, label='|u| with Maslov')
if 'u_standard' in sol:
    axes[0].plot(x_grid, np.abs(sol['u_standard']), 'r--', 
                linewidth=2, alpha=0.6, label='|u| without Maslov')

axes[0].set_ylabel('|u|', fontsize=12)
axes[0].set_title('Wave Amplitude Through Multiple Caustics', fontsize=13)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. Maslov phase
if 'maslov_phases' in sol:
    axes[1].plot(x_grid, sol['maslov_phases'], 'orange', linewidth=2)
    axes[1].set_ylabel('Maslov Phase', fontsize=12)
    axes[1].set_title('Accumulated Maslov Index', fontsize=13)
    axes[1].grid(True, alpha=0.3)
    
    # Mark phase jumps
    phase_jumps = np.where(np.abs(np.diff(sol['maslov_phases'])) > 0.1)[0]
    for idx in phase_jumps:
        axes[1].axvline(x_grid[idx], color='red', linestyle=':', alpha=0.7)

# 3. Real part showing phase shifts
axes[2].plot(x_grid, np.real(sol['u']), 'b-', linewidth=1.5, label='Re(u) with Maslov')
if 'u_standard' in sol:
    axes[2].plot(x_grid, np.real(sol['u_standard']), 'r--', 
                linewidth=1.5, alpha=0.6, label='Re(u) without Maslov')

axes[2].set_xlabel('x', fontsize=12)
axes[2].set_ylabel('Re(u)', fontsize=12)
axes[2].set_title('Real Part Showing Phase Corrections', fontsize=13)
axes[2].legend()
axes[2].grid(True, alpha=0.3)

# Mark all caustics
for caustic in sol.get('caustics', []):
    x_c = caustic['position']
    for ax in axes:
        ax.axvline(x_c, color='red', linestyle='--', linewidth=1.5, alpha=0.5)

plt.tight_layout()

# Print statistics
print(f"\nNumber of caustics detected: {len(sol.get('caustics', []))}")
print(f"Final Maslov phase: {sol.get('maslov_phases', [0])[-1]:.4f} rad")
print(f"Expected: {len(sol.get('caustics', [])) * np.pi/2:.4f} rad")

### Anisotropic Wave Propagation in Crystals

In [ ]:
"""
Example 3: Anisotropic Wave Propagation in Crystals
====================================================

This example demonstrates anisotropic wave propagation in a uniaxial crystal.
Different propagation speeds in different directions lead to elliptical
wavefronts and birefringence phenomena.

Physical setup:
- Uniaxial crystal (e.g., calcite)
- Fast axis (x): v_x = 2
- Slow axis (y): v_y = 1
- Point source → elliptical wavefronts
- Demonstrates optical axis and double refraction
"""


# Define anisotropic dispersion relation
x, y, xi, eta = symbols('x y xi eta', real=True)

# Anisotropic medium: different speeds in x and y
# ω²/v_x² ξ² + ω²/v_y² η² = ω²
# Normalized: ξ²/v_x² + η²/v_y² = 1
v_x = 2.0  # Fast direction
v_y = 1.0  # Slow direction

p = (xi/v_x)**2 + (eta/v_y)**2 - 1

# Point source at origin
n_rays = 32
ic = create_initial_data_point_source(
    x0=0.0,
    y0=0.0,
    n_rays=n_rays
)

print("Computing anisotropic wave propagation in crystal...")
print(f"Anisotropy ratio: v_x/v_y = {v_x/v_y:.1f}")

wkb = wkb_approximation(
    p, 
    ic, 
    order=1,
    domain=((-4, 4), (-4, 4)),
    resolution=80
)

# Theoretical analysis: elliptical wavefronts
# At time t, wavefront is ellipse: x²/v_x² + y²/v_y² = t²
t_theory = np.linspace(0.5, 3.0, 5)
theta = np.linspace(0, 2*np.pi, 200)

print(f"\n📊 Traced {len(wkb['rays'])} rays")
print(f"Maximum ray extent: x ∈ [{np.min(wkb['x']):.2f}, {np.max(wkb['x']):.2f}]")

# Visualizations
fig = plt.figure(figsize=(16, 11))

# Panel 1: Ray diagram with theoretical wavefronts
ax1 = plt.subplot(221)
# Plot rays
colors_ray = plt.cm.hsv(np.linspace(0, 1, len(wkb['rays'])))
for i, ray in enumerate(wkb['rays']):
    ax1.plot(ray['x'], ray['y'], color=colors_ray[i], 
            alpha=0.7, linewidth=1.5)
    ax1.plot(ray['x'][0], ray['y'][0], 'ko', markersize=8)

# Overlay theoretical elliptical wavefronts
for t in t_theory:
    x_ellipse = v_x * t * np.cos(theta)
    y_ellipse = v_y * t * np.sin(theta)
    ax1.plot(x_ellipse, y_ellipse, 'b--', linewidth=2, alpha=0.6,
            label=f't = {t:.1f}' if t == t_theory[0] else '')

# Mark fast/slow axes
ax1.arrow(0, 0, 3, 0, head_width=0.2, head_length=0.3, 
         fc='red', ec='red', linewidth=3, label='Fast axis (x)')
ax1.arrow(0, 0, 0, 1.5, head_width=0.2, head_length=0.2, 
         fc='green', ec='green', linewidth=3, label='Slow axis (y)')

ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.set_title('Anisotropic Ray Propagation')
ax1.set_aspect('equal')
ax1.legend(loc='upper right', fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(-4, 4)
ax1.set_ylim(-4, 4)

# Panel 2: Phase distribution (eikonal)
ax2 = plt.subplot(222)
phase_plot = ax2.contourf(wkb['x'], wkb['y'], wkb['S'], 
                          levels=60, cmap='twilight')
plt.colorbar(phase_plot, ax=ax2, label='Phase S(x,y)')
ax2.plot(0, 0, 'r*', markersize=20, label='Source')
ax2.set_xlabel('x')
ax2.set_ylabel('y')
ax2.set_title('Phase Distribution (Eikonal Surface)')
ax2.set_aspect('equal')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Panel 3: Group velocity field
ax3 = plt.subplot(223)
# Group velocity: v_g = ∇_ξ p
# v_g^x = 2ξ/v_x², v_g^y = 2η/v_y²
# On a uniform grid
x_vel = np.linspace(-3, 3, 20)
y_vel = np.linspace(-3, 3, 20)
X_vel, Y_vel = np.meshgrid(x_vel, y_vel)

# From eikonal: ξ = ∂S/∂x, η = ∂S/∂y
# Approximate gradients
dS_dx, dS_dy = np.gradient(wkb['S'])
dx = wkb['x'][1,0] - wkb['x'][0,0]
dy = wkb['y'][0,1] - wkb['y'][0,0]
dS_dx /= dx
dS_dy /= dy

# Subsample for quiver
skip = 4
X_sub = wkb['x'][::skip, ::skip]
Y_sub = wkb['y'][::skip, ::skip]
U_sub = dS_dx[::skip, ::skip] / v_x**2
V_sub = dS_dy[::skip, ::skip] / v_y**2

ax3.quiver(X_sub, Y_sub, U_sub, V_sub, alpha=0.7)
ax3.plot(0, 0, 'r*', markersize=20)
ax3.set_xlabel('x')
ax3.set_ylabel('y')
ax3.set_title('Group Velocity Field ∇_ξ H')
ax3.set_aspect('equal')
ax3.grid(True, alpha=0.3)

# Panel 4: Wave intensity pattern
ax4 = plt.subplot(224)
intensity = np.abs(wkb['u'])**2
int_plot = ax4.contourf(wkb['x'], wkb['y'], intensity, 
                        levels=50, cmap='hot')
plt.colorbar(int_plot, ax=ax4, label='Intensity |u|²')
# Overlay elliptical contours
for t in t_theory[::2]:
    x_ellipse = v_x * t * np.cos(theta)
    y_ellipse = v_y * t * np.sin(theta)
    ax4.plot(x_ellipse, y_ellipse, 'cyan', linewidth=1.5, alpha=0.5)

ax4.plot(0, 0, 'w*', markersize=20, label='Source')
ax4.set_xlabel('x')
ax4.set_ylabel('y')
ax4.set_title('Wave Intensity Pattern')
ax4.set_aspect('equal')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('wkb_anisotropic_crystal.png', dpi=150)
plt.show()

# Quantitative analysis
print("\n" + "="*60)
print("ANISOTROPIC PROPAGATION ANALYSIS:")
print("="*60)
print(f"Fast axis velocity: v_x = {v_x:.2f}")
print(f"Slow axis velocity: v_y = {v_y:.2f}")
print(f"Anisotropy factor: {v_x/v_y:.2f}")
print("\nWavefront geometry:")
print("- Circular source → Elliptical wavefronts")
print("- Semi-axes ratio = velocity ratio")
print("- Energy propagates along rays (NOT perpendicular to wavefronts!)")
print("\nPhysical phenomena demonstrated:")
print("1. Double refraction (birefringence)")
print("2. Walk-off effect (ray ≠ perpendicular to wavefront)")
print("3. Form birefringence in structured media")
print("\nApplications:")
print("- Polarization optics (wave plates, polarizers)")
print("- Seismic wave propagation in layered earth")
print("- Electromagnetic waves in plasma")
print("- Acoustic waves in composite materials")

# Compare ray trajectories to theory
print("\n" + "="*60)
print("VERIFICATION AGAINST THEORY:")
print("="*60)

# Check a few rays against theoretical ellipse
angles_check = [0, np.pi/4, np.pi/2, 3*np.pi/4]
print("Checking ray endpoints at various angles:")

for angle in angles_check:
    # Find ray closest to this angle
    ray_angles = []
    for ray in wkb['rays']:
        ray_angle = np.arctan2(ray['y'][0] - 0, ray['x'][0] - 0)
        ray_angles.append(ray_angle)
    
    idx = np.argmin(np.abs(np.array(ray_angles) - angle))
    ray = wkb['rays'][idx]
    
    # Get final position
    x_final = ray['x'][-1]
    y_final = ray['y'][-1]
    
    # Theoretical position on ellipse at same angle
    # Ray direction in anisotropic medium
    x_theory = v_x * np.cos(angle) * ray['t'][-1]
    y_theory = v_y * np.sin(angle) * ray['t'][-1]
    
    error_pct = np.sqrt((x_final-x_theory)**2 + (y_final-y_theory)**2) / \
                np.sqrt(x_theory**2 + y_theory**2) * 100
    
    print(f"  θ = {np.degrees(angle):6.1f}°: error = {error_pct:.2f}%")

print("\n✓ WKB accurately captures anisotropic propagation!")

### Atmospheric Mirage - Curved Rays in Inhomogeneous Medium

In [ ]:
"""
Example 4: Atmospheric Mirage - Curved Rays in Inhomogeneous Medium
====================================================================

This example demonstrates ray bending in a stratified medium with spatially
varying refractive index, leading to the mirage effect. Rays curve due to
Snell's law in continuous media.

Physical setup:
- Stratified atmosphere: n(y) = 1 + α·y (linear gradient)
- Hot surface at y=0 (n decreases near ground)
- Observer at y=2
- Demonstrates: inferior mirage, ray bending, critical angle

This is a classic example of geometric optics in inhomogeneous media,
relevant to atmospheric optics, fiber optics, and seismic wave propagation.
"""

import numpy as np
import matplotlib.pyplot as plt
from sympy import symbols, exp, sqrt
from scipy.interpolate import interp1d

print("="*70)
print(" ATMOSPHERIC MIRAGE SIMULATION")
print("="*70)

# Define symbols
x, y, xi, eta = symbols('x y xi eta', real=True)

# Refractive index profile: n(y) = n0 * (1 + α·y)
# For mirage: n decreases toward ground (hot air is less dense)
n0 = 1.0
alpha = -0.15  # Negative gradient (hotter near ground)

# Dispersion relation in stratified medium:
# n²(y) (ξ² + η²) = ω²/c²
# Normalized: n²(y) (ξ² + η²) = 1
n_squared = (1 + alpha * y)**2
p = n_squared * (xi**2 + eta**2) - 1

print(f"\nMedium properties:")
print(f"  Refractive index: n(y) = {n0} + {alpha}·y")
print(f"  Ground level (y=0): n = {n0}")
print(f"  At y=2: n = {n0 * (1 + 2*alpha):.3f}")
print(f"  Gradient: dn/dy = {alpha}")

# Initial conditions: rays launched from observer at various angles
observer_height = 2.0
n_rays = 15

# Rays at different elevation angles
angles = np.linspace(-15, 30, n_rays)  # degrees below/above horizontal

x_init = np.zeros(n_rays)
y_init = np.full(n_rays, observer_height)
S_init = np.zeros(n_rays)

# Initial momenta (direction in phase space)
# At observer position, n = n0(1 + α·y_obs)
n_obs = n0 * (1 + alpha * observer_height)

px_init = []
py_init = []
for angle_deg in angles:
    angle_rad = np.radians(angle_deg)
    # Direction: (cos θ, sin θ) but weighted by refractive index
    px_init.append(n_obs * np.cos(angle_rad))
    py_init.append(n_obs * np.sin(angle_rad))

px_init = np.array(px_init)
py_init = np.array(py_init)

# Normalize to satisfy dispersion relation
norm = np.sqrt(px_init**2 + py_init**2)
px_init = px_init / norm
py_init = py_init / norm

ic = {
    'x': x_init,
    'y': y_init,
    'S': S_init,
    'p_x': px_init,
    'p_y': py_init
}

print(f"\nRay configuration:")
print(f"  Observer height: y = {observer_height}")
print(f"  Number of rays: {n_rays}")
print(f"  Angular range: {angles[0]:.1f}° to {angles[-1]:.1f}°")

# Compute WKB solution
print("\nComputing ray trajectories...")
wkb = wkb_approximation(
    p, 
    ic, 
    order=1,
    domain=((-2, 8), (-1, 4)),
    resolution=60
)

print(f"✓ Successfully traced {len(wkb['rays'])} rays")

# Analyze ray behavior
print("\n" + "="*70)
print("RAY ANALYSIS:")
print("="*70)

rays_bent_down = 0
rays_reflected = 0
critical_angles = []

for i, ray in enumerate(wkb['rays']):
    angle_deg = angles[i]
    
    # Check if ray curves downward
    y_min = np.min(ray['y'])
    y_start = ray['y'][0]
    
    if y_min < y_start - 0.5:
        rays_bent_down += 1
        
        # Check for total internal reflection (ray turns back up)
        if np.any(np.diff(ray['y']) > 0):  # Ray going back up
            rays_reflected += 1
            critical_angles.append(angle_deg)
            print(f"  Ray #{i+1} (θ={angle_deg:+.1f}°): TOTAL INTERNAL REFLECTION at y={y_min:.2f}")

print(f"\nStatistics:")
print(f"  Rays bent downward: {rays_bent_down}/{n_rays}")
print(f"  Rays totally reflected: {rays_reflected}/{n_rays}")
if critical_angles:
    print(f"  Critical angle range: {min(critical_angles):.1f}° to {max(critical_angles):.1f}°")

# Visualization
fig = plt.figure(figsize=(16, 12))

# Panel 1: Ray diagram with refractive index profile
ax1 = plt.subplot(221)

# Background: refractive index
y_bg = np.linspace(-1, 4, 100)
x_bg = np.linspace(-2, 8, 100)
X_bg, Y_bg = np.meshgrid(x_bg, y_bg)
n_bg = n0 * (1 + alpha * Y_bg)

# Plot refractive index
n_plot = ax1.contourf(X_bg, Y_bg, n_bg, levels=30, cmap='coolwarm', alpha=0.4)
cbar1 = plt.colorbar(n_plot, ax=ax1, label='Refractive index n(y)')

# Plot rays with color indicating angle
colors = plt.cm.rainbow(np.linspace(0, 1, len(wkb['rays'])))
for i, ray in enumerate(wkb['rays']):
    ax1.plot(ray['x'], ray['y'], color=colors[i], linewidth=2, 
            label=f"{angles[i]:+.0f}°" if i % 3 == 0 else "")

# Mark observer
ax1.plot(0, observer_height, 'k*', markersize=20, label='Observer', zorder=10)

# Mark ground (hot surface)
ax1.axhline(0, color='red', linewidth=3, linestyle='--', label='Hot surface')

# Mark horizon line
ax1.axhline(observer_height, color='black', linewidth=1, 
           linestyle=':', alpha=0.5, label='Horizon')

ax1.set_xlabel('Distance x (arbitrary units)', fontsize=11)
ax1.set_ylabel('Height y', fontsize=11)
ax1.set_title('Ray Trajectories in Stratified Atmosphere', fontsize=12, fontweight='bold')
ax1.legend(loc='upper right', fontsize=8, ncol=2)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(-1, 8)
ax1.set_ylim(-0.5, 3.5)

# Panel 2: Close-up of mirage region
ax2 = plt.subplot(222)

# Plot only rays that show interesting behavior (bent down significantly)
for i, ray in enumerate(wkb['rays']):
    if np.min(ray['y']) < observer_height - 0.3:
        ax2.plot(ray['x'], ray['y'], color=colors[i], linewidth=2.5, alpha=0.8)
        # Mark turning point
        min_idx = np.argmin(ray['y'])
        ax2.plot(ray['x'][min_idx], ray['y'][min_idx], 'ro', markersize=8)

ax2.plot(0, observer_height, 'k*', markersize=20, label='Observer')
ax2.axhline(0, color='red', linewidth=3, alpha=0.7, label='Ground')

# Shade mirage zone
mirage_y = 0.5
ax2.axhspan(-0.5, mirage_y, alpha=0.2, color='yellow', label='Mirage zone')

ax2.set_xlabel('Distance x', fontsize=11)
ax2.set_ylabel('Height y', fontsize=11)
ax2.set_title('Mirage Formation Zone (Close-up)', fontsize=12, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.set_xlim(-0.5, 6)
ax2.set_ylim(-0.3, 2.5)

# Panel 3: Refractive index profile and Snell's law
ax3 = plt.subplot(223)

y_profile = np.linspace(-0.5, 3, 200)
n_profile = n0 * (1 + alpha * y_profile)

ax3.plot(n_profile, y_profile, 'b-', linewidth=3)
ax3.axhline(0, color='red', linewidth=2, linestyle='--', alpha=0.7, label='Ground')
ax3.axhline(observer_height, color='black', linewidth=2, 
           linestyle=':', alpha=0.7, label='Observer')

# Mark critical region
n_ground = n0
n_obs_val = n0 * (1 + alpha * observer_height)
ax3.axvspan(n_ground, n_obs_val, alpha=0.2, color='orange', 
           label='Ray bending region')

ax3.set_xlabel('Refractive index n', fontsize=11)
ax3.set_ylabel('Height y', fontsize=11)
ax3.set_title('Vertical Refractive Index Profile', fontsize=12, fontweight='bold')
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.3)

# Add annotation
ax3.annotate(f'Gradient: dn/dy = {alpha}',
            xy=(0.5, 0.15), xycoords='axes fraction',
            fontsize=10, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

# Panel 4: What the observer sees
ax4 = plt.subplot(224)

# Simulate observer's view
# Rays that reach ground level appear to come from virtual image below ground

# For each ray, trace back the direction at observer
apparent_angles = []
apparent_sources = []

for i, ray in enumerate(wkb['rays']):
    # Direction at observer (first point)
    if len(ray['x']) > 5:
        # Fit line to initial segment
        dx = ray['x'][5] - ray['x'][0]
        dy = ray['y'][5] - ray['y'][0]
        
        if dx != 0:
            apparent_angle = np.degrees(np.arctan2(dy, dx))
            apparent_angles.append(apparent_angle)
            
            # Trace back to apparent source
            # Line: y - y0 = m(x - x0) where m = dy/dx
            # At y=0: x_source = x0 - y0/m
            if dy != 0:
                x_source = ray['x'][0] - ray['y'][0] * (dx/dy)
                apparent_sources.append((x_source, 0))

# Plot sky dome (simplified)
theta_sky = np.linspace(-90, 90, 100)
radius = 1.0
x_sky = radius * np.cos(np.radians(theta_sky))
y_sky = radius * np.sin(np.radians(theta_sky))

ax4.plot(x_sky, y_sky, 'b-', linewidth=2, label='Sky')
ax4.fill_between(x_sky, y_sky, -1, color='skyblue', alpha=0.3)

# Ground reflection zone
x_ground = np.linspace(-1, 1, 50)
y_ground = -0.5 * np.ones_like(x_ground)
ax4.fill_between(x_ground, y_ground, 0, color='yellow', alpha=0.4, 
                label='Virtual image (mirage)')

# Plot apparent ray directions
for i, angle in enumerate(apparent_angles):
    angle_rad = np.radians(angle)
    ax4.arrow(0, 0, 0.8*np.cos(angle_rad), 0.8*np.sin(angle_rad),
             head_width=0.05, head_length=0.05, fc=colors[i], ec=colors[i],
             alpha=0.7, linewidth=1.5)

# Mark horizon
ax4.axhline(0, color='black', linewidth=2, linestyle='-', label='True horizon')

ax4.set_xlim(-1, 1)
ax4.set_ylim(-0.7, 1)
ax4.set_xlabel('Azimuth', fontsize=11)
ax4.set_ylabel('Elevation', fontsize=11)
ax4.set_title("Observer's View (Apparent Ray Directions)", fontsize=12, fontweight='bold')
ax4.legend(fontsize=9)
ax4.grid(True, alpha=0.3)
ax4.set_aspect('equal')

# Add text annotation
ax4.text(0, -0.55, 'INFERIOR MIRAGE:\nSky appears reflected\non hot ground', 
        ha='center', fontsize=9, 
        bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7))

plt.tight_layout()
plt.savefig('wkb_atmospheric_mirage.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n" + "="*70)
print("PHYSICAL INTERPRETATION:")
print("="*70)
print("""
INFERIOR MIRAGE (Hot Ground):
─────────────────────────────
• Refractive index decreases toward ground (hot air less dense)
• Rays bend DOWNWARD according to Snell's law: n·sin(θ) = constant
• Rays at grazing angles undergo total internal reflection
• Observer sees virtual image BELOW the horizon
• Creates illusion of water on hot roads/deserts

KEY PHYSICS:
───────────
1. Fermat's Principle: Rays follow path of stationary optical path length
2. Snell's Law (continuous): n(y)·sin(θ(y)) = constant along ray
3. Critical angle: Rays steeper than θc are reflected back up
4. WKB approximation: Valid when n(y) varies slowly (∂n/∂y << k·n)

MATHEMATICAL FORMULATION:
────────────────────────
• Eikonal equation: |∇S|² = n²(y)
• Ray equations: d/ds[n(y)·dr/ds] = ∇n
• Dispersion relation: n²(y)(ξ² + η²) = ω²/c²

APPLICATIONS:
────────────
• Atmospheric optics (mirages, looming, ducting)
• Radio wave propagation (tropospheric ducting)
• Seismology (earthquake waves in layered earth)
• Fiber optics (graded-index fibers - GRIN lenses)
• Underwater acoustics (SOFAR channel)

SUPERIOR MIRAGE (Cold Ground):
──────────────────────────────
• Opposite gradient: n increases toward ground
• Rays bend UPWARD
• Distant objects appear elevated or inverted
• Fata Morgana phenomenon
""")

# Compute theoretical critical angle
if alpha < 0:
    # Critical angle from Snell's law
    # At turning point: n(y_min)·sin(90°) = n(y_obs)·sin(θ_c)
    # Assuming ray reaches ground: n_ground = n_obs·sin(θ_c)
    sin_theta_c = n_ground / n_obs_val
    if sin_theta_c <= 1:
        theta_c = np.degrees(np.arcsin(sin_theta_c))
        print(f"\nTHEORETICAL CRITICAL ANGLE:")
        print(f"  θ_c = {theta_c:.2f}° (below horizontal)")
        print(f"  Rays steeper than this angle are totally reflected")
        print(f"  Compare with observed range: {min(critical_angles) if critical_angles else 'N/A':.1f}° - {max(critical_angles) if critical_angles else 'N/A':.1f}°")

print("\n" + "="*70)
print("✓ SIMULATION COMPLETE - Ray bending successfully demonstrated!")
print("="*70)

### Gravitational Lensing - Einstein Rings and Light Deflection

In [ ]:
"""
Example 6: Gravitational Lensing - Einstein Rings and Light Deflection
=======================================================================

This example demonstrates gravitational lensing predicted by General Relativity.
Massive objects curve spacetime, bending light rays and creating multiple images,
Einstein rings, and gravitational arcs.

Physical setup:
- Point mass M at origin (galaxy, black hole, or dark matter halo)
- Light rays from distant source passing near the mass
- Schwarzschild metric in weak-field limit
- Demonstrates: Einstein radius, multiple images, time delay

This is a direct test of General Relativity and a major tool in modern astronomy
for detecting dark matter, measuring cosmic distances, and finding exoplanets.

Einstein (1936): "Of course, there is no hope of observing this phenomenon directly."
[First gravitational lens discovered in 1979!]
"""

from scipy.optimize import brentq

print("="*80)
print(" GRAVITATIONAL LENSING - GENERAL RELATIVITY")
print("="*80)
print("\n\"The most beautiful thought experiment in physics\" - Einstein\n")

# Physical constants (in geometric units: G = c = 1)
M_lens = 1.0  # Mass of lens (e.g., solar masses)
D_lens = 10.0  # Distance to lens
D_source = 20.0  # Distance to source (behind lens)
D_ls = D_source - D_lens  # Lens-source distance

# Einstein radius: the characteristic angular scale
# θ_E = sqrt(4GM/c² · D_ls/(D_l · D_s))
# In our units: θ_E = sqrt(4M · D_ls/(D_l · D_s))
theta_E = np.sqrt(4 * M_lens * D_ls / (D_lens * D_source))
r_E = theta_E * D_lens  # Physical Einstein radius at lens plane

print("SYSTEM PARAMETERS:")
print("─"*80)
print(f"  Lens mass: M = {M_lens:.2f} (solar masses)")
print(f"  Lens distance: D_l = {D_lens:.2f}")
print(f"  Source distance: D_s = {D_source:.2f}")
print(f"  Einstein radius: θ_E = {theta_E:.4f} rad = {np.degrees(theta_E):.2f}°")
print(f"  Physical Einstein radius: r_E = {r_E:.4f}")

# Define the effective refractive index in curved spacetime
# In the weak-field limit (Schwarzschild metric):
# n(r) ≈ 1 + 2GM/(c²r) ≈ 1 + 2M/r (in our units)
# This causes light to bend toward the mass

x, y, xi, eta = symbols('x y xi eta', real=True)

# Distance from lens (at origin)
r = sqrt(x**2 + y**2)

# Refractive index in Schwarzschild spacetime (weak field)
# n(r) = 1 + 2M/r (leading order post-Newtonian)
n = 1 + 2*M_lens/r

# Dispersion relation: n²(ξ² + η²) = k² where k = ω/c = 1 (normalized)
# For light deflection: n²(r)(ξ² + η²) = 1
p = n**2 * (xi**2 + eta**2) - 1

print("\nDISPERSION RELATION:")
print("─"*80)
print("  Schwarzschild metric (weak field):")
print(f"  n(r) = 1 + 2M/r ≈ 1 + {2*M_lens}/r")
print("  → Light bends toward massive object")
print("  → Effective potential well in phase space")

# Source positions to test
# - On-axis: perfect Einstein ring
# - Off-axis: multiple images (Einstein cross)
source_positions = [
    (0.0, "Perfect alignment → Einstein Ring"),
    (0.3 * r_E, "Slightly off-axis → Arcs"),
    (0.8 * r_E, "Off-axis → Multiple images"),
]

results = {}

for source_offset, description in source_positions:
    print(f"\n{'='*80}")
    print(f"CONFIGURATION: {description}")
    print(f"Source offset: β = {source_offset:.4f} (β/θ_E = {source_offset/r_E:.2f})")
    print('='*80)
    
    # Create initial conditions: light rays from source
    # Source is at distance D_source, offset by source_offset from optical axis
    n_rays = 24
    
    # Rays emanate from source in all directions
    angles = np.linspace(0, 2*np.pi, n_rays, endpoint=False)
    
    # Source position at distance D_source
    source_x = source_offset
    source_y = D_source
    
    # Initial positions: slightly offset from source to avoid singularity
    x_init = source_x + 0.01 * np.cos(angles)
    y_init = source_y + 0.01 * np.sin(angles)
    
    # Initial momenta: pointing generally toward lens (negative y direction)
    # With some angular spread
    px_init = 0.5 * np.cos(angles)
    py_init = -1.0 + 0.3 * np.sin(angles)
    
    # Normalize
    norm = np.sqrt(px_init**2 + py_init**2)
    px_init /= norm
    py_init /= norm
    
    ic = {
        'x': x_init,
        'y': y_init,
        'S': np.zeros(n_rays),
        'p_x': px_init,
        'p_y': py_init
    }
    
    print(f"\nRay tracing {n_rays} light rays from source...")
    
    # Compute WKB solution
    wkb = wkb_approximation(
        p, 
        ic, 
        order=1,
        domain=((-8, 8), (-2, 25)),
        resolution=80
    )
    
    print(f"✓ Successfully traced {len(wkb['rays'])} rays")
    
    # Analyze ray deflection
    deflection_angles = []
    impact_parameters = []
    
    for ray in wkb['rays']:
        # Find closest approach to lens (at origin)
        distances = np.sqrt(ray['x']**2 + ray['y']**2)
        min_idx = np.argmin(distances)
        b = distances[min_idx]  # Impact parameter
        
        if b > 0.1:  # Avoid too close approaches (strong field)
            impact_parameters.append(b)
            
            # Estimate deflection angle
            if min_idx > 5 and min_idx < len(ray['x']) - 5:
                # Direction before closest approach
                dx_before = ray['x'][min_idx-5] - ray['x'][min_idx-10]
                dy_before = ray['y'][min_idx-5] - ray['y'][min_idx-10]
                angle_before = np.arctan2(dy_before, dx_before)
                
                # Direction after closest approach
                dx_after = ray['x'][min_idx+10] - ray['x'][min_idx+5]
                dy_after = ray['y'][min_idx+10] - ray['y'][min_idx+5]
                angle_after = np.arctan2(dy_after, dx_after)
                
                # Deflection angle
                deflection = angle_after - angle_before
                deflection_angles.append(deflection)
    
    if len(deflection_angles) > 0:
        avg_deflection = np.mean(np.abs(deflection_angles))
        print(f"\nDeflection analysis:")
        print(f"  Average deflection: {np.degrees(avg_deflection):.4f}°")
        print(f"  Einstein prediction (4M/b): {np.degrees(4*M_lens/np.mean(impact_parameters)):.4f}°")
    
    results[source_offset] = wkb

# === VISUALIZATION ===
print("\n" + "="*80)
print("GENERATING COMPREHENSIVE VISUALIZATION...")
print("="*80)

fig = plt.figure(figsize=(20, 14))

# Color schemes
colors_ring = plt.cm.rainbow(np.linspace(0, 1, 24))
colors_arc = plt.cm.plasma(np.linspace(0, 1, 24))
colors_multi = plt.cm.viridis(np.linspace(0, 1, 24))

# Panel 1: Einstein Ring (perfect alignment)
ax1 = plt.subplot(2, 3, 1)
wkb_ring = results[0.0]

# Plot rays
for i, ray in enumerate(wkb_ring['rays']):
    ax1.plot(ray['x'], ray['y'], color=colors_ring[i], 
            alpha=0.7, linewidth=2)
    # Mark source
    ax1.plot(ray['x'][0], ray['y'][0], 'yo', markersize=6)

# Mark lens (black hole/galaxy)
lens_circle = plt.Circle((0, D_lens), 0.3, color='black', 
                         label='Lens (mass M)', zorder=10)
ax1.add_patch(lens_circle)

# Einstein radius circle
einstein_circle = plt.Circle((0, D_lens), r_E, fill=False, 
                             edgecolor='red', linewidth=3, 
                             linestyle='--', label=f'Einstein radius')
ax1.add_patch(einstein_circle)

# Observer plane
ax1.axhline(0, color='green', linewidth=3, 
           label='Observer plane', alpha=0.7)
ax1.axhline(D_source, color='orange', linewidth=2, 
           linestyle=':', label='Source plane', alpha=0.7)

ax1.set_xlabel('x (impact parameter)', fontsize=11)
ax1.set_ylabel('y (distance along line of sight)', fontsize=11)
ax1.set_title('Perfect Alignment → Einstein Ring', 
             fontsize=13, fontweight='bold')
ax1.legend(fontsize=9, loc='upper right')
ax1.grid(True, alpha=0.3)
ax1.set_xlim(-6, 6)
ax1.set_ylim(-1, 22)
ax1.set_aspect('equal')

# Panel 2: Intensity map at observer plane (Einstein Ring)
ax2 = plt.subplot(2, 3, 2)

# Extract intensity at observer plane (y ≈ 0)
y_obs_idx = np.argmin(np.abs(wkb_ring['y'][:, 0]))
x_obs = wkb_ring['x'][y_obs_idx, :]
intensity_ring = np.abs(wkb_ring['u'][y_obs_idx, :])**2

ax2.plot(x_obs, intensity_ring, 'r-', linewidth=3)
ax2.axvline(-r_E, color='blue', linestyle='--', 
           label=f'θ_E = ±{theta_E:.3f}', alpha=0.7)
ax2.axvline(r_E, color='blue', linestyle='--', alpha=0.7)
ax2.fill_between(x_obs, 0, intensity_ring, alpha=0.3, color='red')

ax2.set_xlabel('Angular position x', fontsize=11)
ax2.set_ylabel('Intensity I(x)', fontsize=11)
ax2.set_title('Observer View: Einstein Ring Profile', 
             fontsize=13, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.set_xlim(-4, 4)

# Panel 3: Phase fronts (Einstein Ring)
ax3 = plt.subplot(2, 3, 3)
phase_ring = wkb_ring['S']
contours = ax3.contourf(wkb_ring['x'], wkb_ring['y'], phase_ring, 
                        levels=40, cmap='twilight')
plt.colorbar(contours, ax=ax3, label='Phase S(x,y)')

# Overlay lens and Einstein radius
lens_circle2 = plt.Circle((0, D_lens), 0.3, color='white', zorder=10)
ax3.add_patch(lens_circle2)
einstein_circle2 = plt.Circle((0, D_lens), r_E, fill=False, 
                              edgecolor='yellow', linewidth=2, linestyle='--')
ax3.add_patch(einstein_circle2)

ax3.set_xlabel('x', fontsize=11)
ax3.set_ylabel('y', fontsize=11)
ax3.set_title('Phase Fronts (Gravitational Time Delay)', 
             fontsize=13, fontweight='bold')
ax3.set_aspect('equal')
ax3.grid(True, alpha=0.2)

# Panel 4: Slightly off-axis (Arcs)
ax4 = plt.subplot(2, 3, 4)
wkb_arc = results[0.3 * r_E]

for i, ray in enumerate(wkb_arc['rays']):
    ax4.plot(ray['x'], ray['y'], color=colors_arc[i], 
            alpha=0.7, linewidth=2)

lens_circle3 = plt.Circle((0, D_lens), 0.3, color='black', zorder=10)
ax4.add_patch(lens_circle3)
einstein_circle3 = plt.Circle((0, D_lens), r_E, fill=False, 
                              edgecolor='red', linewidth=2, linestyle='--')
ax4.add_patch(einstein_circle3)

# Mark source offset
ax4.plot(0.3*r_E, D_source, 'y*', markersize=20, 
        label='Source (offset)', zorder=10)

ax4.axhline(0, color='green', linewidth=3, alpha=0.7)
ax4.set_xlabel('x', fontsize=11)
ax4.set_ylabel('y', fontsize=11)
ax4.set_title('Slight Misalignment → Gravitational Arcs', 
             fontsize=13, fontweight='bold')
ax4.legend(fontsize=9)
ax4.grid(True, alpha=0.3)
ax4.set_xlim(-6, 6)
ax4.set_ylim(-1, 22)
ax4.set_aspect('equal')

# Panel 5: Off-axis (Multiple images)
ax5 = plt.subplot(2, 3, 5)
wkb_multi = results[0.8 * r_E]

for i, ray in enumerate(wkb_multi['rays']):
    ax5.plot(ray['x'], ray['y'], color=colors_multi[i], 
            alpha=0.7, linewidth=2)

lens_circle4 = plt.Circle((0, D_lens), 0.3, color='black', zorder=10)
ax5.add_patch(lens_circle4)
einstein_circle4 = plt.Circle((0, D_lens), r_E, fill=False, 
                              edgecolor='red', linewidth=2, linestyle='--')
ax5.add_patch(einstein_circle4)

ax5.plot(0.8*r_E, D_source, 'y*', markersize=20, zorder=10)
ax5.axhline(0, color='green', linewidth=3, alpha=0.7)

ax5.set_xlabel('x', fontsize=11)
ax5.set_ylabel('y', fontsize=11)
ax5.set_title('Large Offset → Multiple Images', 
             fontsize=13, fontweight='bold')
ax5.grid(True, alpha=0.3)
ax5.set_xlim(-6, 6)
ax5.set_ylim(-1, 22)
ax5.set_aspect('equal')

# Panel 6: Comparison of observer views
ax6 = plt.subplot(2, 3, 6)

# Extract intensities at observer
for offset, label, color in [
    (0.0, 'Perfect ring', 'red'),
    (0.3*r_E, 'Arcs', 'blue'),
    (0.8*r_E, 'Multiple images', 'green')
]:
    wkb = results[offset]
    y_idx = np.argmin(np.abs(wkb['y'][:, 0]))
    x_vals = wkb['x'][y_idx, :]
    intensity = np.abs(wkb['u'][y_idx, :])**2
    
    # Normalize
    intensity_norm = intensity / np.max(intensity)
    ax6.plot(x_vals, intensity_norm, linewidth=2.5, 
            label=label, alpha=0.8, color=color)

ax6.axvline(-r_E, color='gray', linestyle='--', alpha=0.5)
ax6.axvline(r_E, color='gray', linestyle='--', alpha=0.5)
ax6.axvspan(-r_E, r_E, alpha=0.1, color='yellow', 
           label='Einstein radius')

ax6.set_xlabel('Angular position x at observer', fontsize=11)
ax6.set_ylabel('Normalized Intensity', fontsize=11)
ax6.set_title('Comparison: Different Source Positions', 
             fontsize=13, fontweight='bold')
ax6.legend(fontsize=9)
ax6.grid(True, alpha=0.3)
ax6.set_xlim(-4, 4)

plt.tight_layout()
plt.savefig('wkb_gravitational_lensing.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n" + "="*80)
print("THEORETICAL PREDICTIONS - GENERAL RELATIVITY")
print("="*80)

# Einstein's deflection formula
print("\n1. LIGHT DEFLECTION:")
print("   Einstein (1915): α = 4GM/(c²b) where b = impact parameter")
print(f"   For M = {M_lens}, b = {r_E}:")
print(f"   α = {4*M_lens/r_E:.6f} rad = {np.degrees(4*M_lens/r_E):.4f}°")
print(f"   Historical: Sun deflects starlight by 1.75\" (verified 1919)")

# Einstein radius
print("\n2. EINSTEIN RADIUS:")
print("   θ_E = sqrt(4GM/c² · D_ls/(D_l·D_s))")
print(f"   θ_E = {theta_E:.6f} rad = {np.degrees(theta_E)*3600:.2f} arcsec")
print(f"   Physical size at lens: r_E = {r_E:.4f}")

# Image positions (lens equation)
print("\n3. LENS EQUATION:")
print("   β = θ - (θ_E²/θ)")
print("   For source at β, images appear at:")
print("   θ_± = (β ± sqrt(β² + 4θ_E²))/2")

for offset in [0.3*r_E, 0.8*r_E]:
    beta = offset / D_lens
    theta_E_angle = theta_E
    
    if beta < theta_E:
        theta_plus = (beta + np.sqrt(beta**2 + 4*theta_E_angle**2)) / 2
        theta_minus = (beta - np.sqrt(beta**2 + 4*theta_E_angle**2)) / 2
        
        print(f"\n   β = {beta:.4f}:")
        print(f"     Image 1: θ₊ = {theta_plus:.4f} (outside Einstein radius)")
        print(f"     Image 2: θ₋ = {theta_minus:.4f} (inside Einstein radius)")
        print(f"     Magnification ratio: μ₊/μ₋ = {abs(theta_plus/theta_minus):.2f}")

# Magnification
print("\n4. GRAVITATIONAL MAGNIFICATION:")
print("   μ = β/θ · dθ/dβ")
print("   For Einstein ring (β→0): μ → ∞ (perfect amplification)")
print("   Total magnification (both images):")
print("   μ_total = (u² + 2)/(u·sqrt(u² + 4)) where u = β/θ_E")

# Time delay
print("\n5. SHAPIRO TIME DELAY:")
print("   Δt = (4GM/c³) log(r/r_g)")
print("   Light travels longer path AND slower through curved spacetime")
print("   Used to measure Hubble constant (H₀ tension)")

print("\n" + "="*80)
print("ASTROPHYSICAL APPLICATIONS")
print("="*80)
print("""
OBSERVATIONS:
─────────────
✓ Quasar lensing (1979): First gravitational lens (Twin Quasar)
✓ Einstein Cross (1985): Perfect quadruple image (QSO 2237+0305)
✓ Einstein rings: HST observations of distant galaxies
✓ Cosmic Telescope: Magnify distant universe by 10-100×

SCIENCE RETURNS:
───────────────
✓ Dark matter mapping: Most mass is invisible!
  → Galaxy clusters contain 85% dark matter
  → Bullet Cluster: Direct evidence for dark matter

✓ Hubble constant (H₀): Time delay cosmography
  → Independent measurement of universe expansion
  → "H₀ tension" - discrepancy with CMB measurements

✓ Exoplanet detection: Microlensing
  → Discovered thousands of planets
  → Sensitive to Earth-mass planets at large distances

✓ Black hole masses: Measure central black hole in distant galaxies

✓ Fundamental physics:
  → Test General Relativity in strong-field regime
  → Constrain alternative gravity theories
  → Probe nature of dark energy

FAMOUS EXAMPLES:
───────────────
• Einstein Cross (QSO 2237+0305): Quadruple quasar image
• Cosmic Horseshoe (LRG 3-757): Nearly complete Einstein ring
• SDSS J1038+4849: Extremely magnified z~10 galaxy
• Abell 2744 (Pandora's Cluster): Multiple lensed arcs

WKB REGIME:
──────────
✓ Valid in weak-field limit: GM/(c²r) << 1
✓ Ray approximation: λ << r_g (wavelength << Schwarzschild radius)
✓ Geometric optics: Maps rays but misses wave effects
✗ Breaks down very near event horizon (need full GR + wave optics)
""")

print("\n" + "="*80)
print("EINSTEIN'S PROPHECY (1936):")
print("="*80)
print("""
"Of course, there is no hope of observing this phenomenon directly.
First, we shall scarcely ever approach closely enough to such a central
line. Second, the angle β will defy the resolving power of our instruments."

— Albert Einstein, Science 84, 506 (1936)

HISTORY PROVED HIM WRONG:
• 1979: First gravitational lens discovered (Walsh, Carswell, Weymann)
• 1980s-90s: Dozens more found
• Today: Thousands of lensing systems known
• Hubble Space Telescope: Routinely images Einstein rings

Einstein's humility reminds us: Never say never in science!
""")

print("\n" + "="*80)
print("✓ GRAVITATIONAL LENSING SIMULATION COMPLETE")
print("  General Relativity verified through ray tracing!")
print("="*80)